#baseline

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from google.colab import drive
drive.mount('/content/drive')

# 이미지 로드 및 변환 (RGB 이미지)
image = cv2.imread('/content/drive/My Drive/Colab Notebooks/4611.png', cv2.IMREAD_COLOR)
image_original = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def psnr(img1, img2):
    assert img1.shape == img2.shape, "이미지 크기 다름"
    if img1.dtype == np.float32 or img1.dtype == np.float64:
        max_pixel = 1.0
    else:
        max_pixel = 255.0
    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

In [ ]:
# 준비물 만들기
def downsampling(image):
    image = image.astype(np.float32)
    H, W, C = image.shape
    sigma, kernel_size = 1, 3

    # 가우시안 커널 만들기
    ax = np.arange(-kernel_size//2 , kernel_size//2 + 1) # 커널 사이즈 정의
    xx, yy = np.meshgrid(ax, ax)                         # 좌표 그리드 생성
    kernel = np.exp(-(xx**2 + yy**2) / (2. * sigma**2))  # 가우시안 커널 생성
    kernel = kernel / np.sum(kernel)                     # 정규화

    # blurring
    k = kernel.shape[0]
    pad = k // 2
    padded_img = np.pad(image, ((pad, pad), (pad, pad), (0, 0)), mode='reflect')
    blurred = np.zeros_like(image, dtype=np.float32)

    for y in range(H):
        for x in range(W):
            for c in range(C):
                region = padded_img[y:y+k, x:x+k, c]
                blurred[y, x, c] = np.sum(region * kernel) # blur

    # 2x2 평균 다운샘플링
    H, W, C = blurred.shape
    blurred = blurred[:H - H % 2, :W - W % 2, :]
    downsampled = blurred.reshape(H//2, 2, W//2, 2, C).mean(axis=(1, 3))
    return downsampled.astype(np.uint8)

In [ ]:
# 랜덤 영역 선택
def random_choose(image, patch_size):
    H, W, C = image.shape
    n_h = H // patch_size
    n_w = W // patch_size
    total_patches = n_h * n_w
    k = 2**14 // (patch_size * patch_size)

    # 전체 패치 인덱스 리스트 생성
    patch_indices = [(i, j) for i in range(n_h) for j in range(n_w)]
    selected = np.random.choice(len(patch_indices), k, replace=False)
    selected_patches = [patch_indices[i] for i in selected]
    # 출력용 이미지 (모든 픽셀 0)
    output = np.zeros_like(image)
    # 선택된 패치만 복사
    for i, j in selected_patches:
        y, x = i * patch_size, j * patch_size
        output[y:y+patch_size, x:x+patch_size, :] = image[y:y+patch_size, x:x+patch_size, :]
    return output

In [ ]:
# interpolation

#Baseline
def bilinear(image):
    H, W, C = image.shape
    new_H, new_W = H * 2, W * 2
    upsampled = np.zeros((new_H, new_W, C), dtype=image.dtype)

    # 업샘플된 이미지의 픽셀 순회
    for y in range(new_H):   # 0부터 new_H-1)까지
        for x in range(new_W):
            # 원래 이미지에서 어떤 위치인지 계산
            src_y = y / 2.0
            src_x = x / 2.0
            # 해당 위치 주변 4픽셀 좌표 구하기
            y0 = int(np.floor(src_y))
            x0 = int(np.floor(src_x))
            y1 = min(y0 + 1, H - 1)
            x1 = min(x0 + 1, W - 1)

            dy = src_y - y0
            dx = src_x - x0

            for c in range(C):
                top = (1 - dx) * image[y0, x0, c] + dx * image[y0, x1, c]
                bottom = (1 - dx) * image[y1, x0, c] + dx * image[y1, x1, c]
                value = (1 - dy) * top + dy * bottom
                upsampled[y, x, c] = np.clip(value, 0, 255)

    return upsampled.astype(np.uint8)

# 최종 이미지 생성
def restoration1(image1, image2):
    image1_up = bilinear(image1).astype(np.uint8)
    mask = np.all(image2 == 0, axis=-1)
    output = image2.copy()
    output[mask] = image1_up[mask]
    return output

# our interpolation

In [ ]:
################### 나만의 interpolation 1, backward process bicubic interpolation ###################
def cubic_weight(t):
    t = abs(t)
    if t <= 1:
        return (1.5 * t - 2.5) * t * t + 1
    elif t < 2:
        return ((-0.5 * t + 2.5) * t - 4) * t + 2
    else:
        return 0

def bicubic_upsampling(image):
    H, W, C = image.shape
    new_H, new_W = H * 2, W * 2
    upsampled = np.zeros((new_H, new_W, C), dtype=np.float32)

    for y in range(new_H):
        for x in range(new_W):
            src_y = y / 2.0
            src_x = x / 2.0

            y_int = int(np.floor(src_y))
            x_int = int(np.floor(src_x))

            for c in range(C):
                value = 0.0
                for m in range(-1, 3):
                    for n in range(-1, 3):
                        yy = np.clip(y_int + m, 0, H - 1)
                        xx = np.clip(x_int + n, 0, W - 1)

                        wy = cubic_weight(src_y - (y_int + m))
                        wx = cubic_weight(src_x - (x_int + n))

                        value += image[yy, xx, c] * wy * wx

                upsampled[y, x, c] = np.clip(value, 0, 255)

    return upsampled.astype(np.uint8)

# 큰 이미지 합치기
def restoration2(image1, image2):
    image1_up = bicubic_upsampling(image1).astype(np.uint8)
    mask = np.all(image2 == 0, axis=-1)
    output = image2.copy()
    output[mask] = image1_up[mask]
    return output

In [ ]:
################### 나만의 interpolation 2, forward process interpolation ###################
def expand(image):
    H, W, C = image.shape
    img_sparse = np.zeros((H * 2, W * 2, C), dtype=image.dtype)
    img_sparse[::2, ::2] = image  # 2칸마다 픽셀 배치
    return img_sparse

def assemble(image1, img_sparse):
    # 비어 있는 픽셀 위치 마스크 (RGB 전체가 0이면 empty 판단)
    mask = np.all(image1 == 0, axis=2)  # binary mask
    result = image1.copy()
    # mask가 True인 위치에만 sparse 이미지의 값을 복사
    result[mask] = img_sparse[mask]
    return result

def restoration3(image):
    img = image.copy().astype(np.float32)
    H, W, C = img.shape
    filled = img.copy()
    mask = np.any(img != 0, axis=2)  # axis=2, 0이 아닌 rgb

    for y in range(H):
        for x in range(W):
            if not mask[y, x]:  # 해당 픽셀에 값이 없으면
                vals = []
                weights = []

                for dy in [-1, 0, 1]:
                    for dx in [-1, 0, 1]:
                        ny, nx = y + dy, x + dx
                        if (0 <= ny < H) and (0 <= nx < W) and mask[ny, nx]: # 이웃한 픽셀 값이 유효할 때만
                            dist = max(abs(dy), abs(dx))                     # 있으면 1
                            weight = 1.0 / (dist + 1e-5)                     # 가까울수록 가중치 up
                            vals.append(img[ny, nx])
                            weights.append(weight)

                if vals:
                    vals = np.array(vals)
                    weights = np.array(weights).reshape(-1, 1)
                    weighted_avg = np.sum(vals * weights, axis=0) / np.sum(weights)
                    filled[y, x] = weighted_avg

    return np.clip(filled, 0, 255).astype(np.uint8)

# our region sensing


In [ ]:
################################################################### 주파수 영역 선택 ############################################################
def fft_spectrum_energy(patch):
    ratio=0.25
    f = np.fft.fft2(patch)
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)

    H, W = magnitude.shape
    cy, cx = H // 2, W // 2

    # 중심(저주파) 영역을 False(제외) 처리
    low = int(min(H, W) * ratio / 2)
    mask = np.ones_like(magnitude, dtype=bool)
    mask[cy - low:cy + low, cx - low:cx + low] = False

    high_freq_energy = np.std(magnitude[mask])                    # sum / std
    return high_freq_energy

def frequency_choose(image,target,grid_size):
    H, W, _ = image.shape
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # 크기와 index
    energies = []
    positions = []
    k = 2**14 // (grid_size * grid_size)
    for y in range(0, H, grid_size):
        for x in range(0, W, grid_size):
            patch = gray_image[y:y+grid_size, x:x+grid_size]
            energy = fft_spectrum_energy(patch)
            energies.append(energy)
            positions.append((y, x))

    energies = np.array(energies)
    positions = np.array(positions)

    # 에너지 큰 순서로 상위 64개 선택
    top_idx = np.argsort(energies)[-k:]
    masked_image = np.zeros_like(image)

    for idx in top_idx:
        y, x = positions[idx]
        masked_image[y:y+grid_size, x:x+grid_size] = target[y:y+grid_size, x:x+grid_size]

    return masked_image

In [ ]:
################################################################### Gradient ############################################################
def gradient_choose(image,target,grid_size):
    H, W, C = image.shape
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 전체 gradient magnitude 계산
    gray_padded = np.pad(gray, pad_width=1, mode='reflect')  # (H+2, W+2)

    # 2. x, y 방향 gradient 계산 (shift 방식)
    gx = np.abs(gray_padded[1:-1, 2:] - gray_padded[1:-1, 1:-1])  # (H, W)
    gy = np.abs(gray_padded[2:, 1:-1] - gray_padded[1:-1, 1:-1])  # (H, W)

    # 3. 전체 gradient magnitude 계산
    grad_mag = np.sqrt(gx**2 + gy**2)  # (H, W)

    gradients = []
    positions = []
    k = 2**14 // (grid_size * grid_size)

    for y in range(0, H, grid_size):
        for x in range(0, W, grid_size):
            patch_grad = grad_mag[y:y+grid_size, x:x+grid_size]
            mean_grad = np.sum(patch_grad)  # sum,std
            gradients.append(mean_grad)
            positions.append((y, x))

    gradients = np.array(gradients)
    positions = np.array(positions)

    top_idx = np.argsort(gradients)[-k:]
    masked_image = np.zeros_like(image)

    for idx in top_idx:
        y, x = positions[idx]
        masked_image[y:y+grid_size, x:x+grid_size] = target[y:y+grid_size, x:x+grid_size]

    return masked_image

In [ ]:
################################################ Gaussian Laplacian Pyramid choose ####################################
def laplacian_choose(image,target,grid_size):
    H, W, C = image.shape

    image_down = downsampling(image)
    image_up = bilinear(image_down)

    L1 = np.abs(image-image_up)
    L1 =  cv2.cvtColor(L1, cv2.COLOR_BGR2GRAY)

    scores = []
    positions = []
    k = 2**14 // (grid_size * grid_size)
    for y in range(0, H, grid_size):
        for x in range(0, W, grid_size):
            patch = L1[y:y+grid_size, x:x+grid_size]
            score = np.std(patch)                           #평균, 표준편차
            #score = np.sum(patch)
            scores.append(score)
            positions.append((y, x))

    scores = np.array(scores)
    positions = np.array(positions)

    top_idx = np.argsort(scores)[-k:]
    masked_image = np.zeros_like(image)

    for idx in top_idx:
        y, x = positions[idx]
        masked_image[y:y+grid_size, x:x+grid_size] = target[y:y+grid_size, x:x+grid_size]

    return masked_image

In [ ]:
# 준비물(원본): 512*512, 256*256, 128*128
image_512 = image_original
image_256 = downsampling(image_original)
image_128 = downsampling(image_256)
image_high_up = bilinear(image_128)

In [ ]:
# baseline
K_base = 4
# 영역선택
image_high = image_128
image_mid = random_choose(image_256,K_base)
image_low = random_choose(image_512,K_base)
# 선택한 영역 + 보간
image_re_256 = restoration1(image_high,image_mid)
image_re_512 = restoration1(image_re_256,image_low)

p1=psnr(image_256,image_re_256)
p2=psnr(image_512,image_re_512)

In [ ]:
print(p1,p2)

29.417593645881812 24.383815230102577


In [ ]:
K_base = 4
image_high = image_128
image_mid = random_choose(image_256,K_base)
image_low = random_choose(image_512,K_base)

g1= gradient_choose(image_high_up,image_256,K_base)

# 나만의 interpolation2
img_sparse = expand(image_128)
img_combined = assemble(g1, img_sparse)
image_re_256_m2 = restoration3(img_combined)
image_re_256_m2_up = bilinear(image_re_256_m2)

img_sparse2 = expand(image_re_256_m2)
g2 = gradient_choose(image_re_256_m2_up,image_512,K_base)
img_combined2 = assemble(image_low, img_sparse2)
image_re_512_m2 = restoration3(img_combined2)

p1=psnr(image_256,image_re_256_m2)
p2=psnr(image_512,image_re_512_m2)

print(p1)
print(p2)

32.68298407498561
25.805439879296344
